<a href="https://colab.research.google.com/github/nurallyssaroslan/KD344403-S2-25-26-G8-PROJECT/blob/main/milestone1_datapipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Initial Exploration**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('/content/salaries.csv')
display(df.head())

In [ ]:
pip install ydata-profiling

In [ ]:
from ydata_profiling import ProfileReport

profile = ProfileReport(df, title="Pandas Profiling Report")
profile.to_notebook_iframe()

**Data Cleaning**

In [ ]:
print("Size of df (rows, columns):")
display(df.shape)

In [ ]:
print("Number of duplicate rows in df:")
display(df.duplicated().sum())

In [ ]:
df = df.drop_duplicates()

In [ ]:
print("Size of df (rows, columns):")
display(df.shape)

In [ ]:
# drop redundant columns
df = df.drop(columns=['salary', 'salary_currency'])

In [ ]:
display(df.head())

In [ ]:
print("Size of df (rows, columns):")
display(df.shape)

In [ ]:
df['job_title'] = df['job_title'].str.lower()
df['experience_level'] = df['experience_level'].str.lower()
df['employment_type'] = df['employment_type'].str.lower()
df['employee_residence'] = df['employee_residence'].str.lower()
df['company_location'] = df['company_location'].str.lower()
df['company_size'] = df['company_size'].str.lower()

display(df.head())

In [ ]:
# checking how remote ratio influences the target
plt.figure(figsize=(10, 6))
sns.boxplot(x='remote_ratio', y='salary_in_usd', data=df)
plt.title('Salary Distribution by Remote Ratio')
plt.xlabel('Remote Ratio (%)')
plt.ylabel('Salary in USD')
plt.grid(axis='y', alpha=0.75)
plt.show()

# the salary distribution between fully remote and on-site are almost similar
# this may indicate that fully remote and on-site does not have a huge effect on the target
# however, hybrid suggest some variation in salary

In [ ]:
remote_ratio_percentages = df['remote_ratio'].value_counts(normalize=True) * 100
display(remote_ratio_percentages.round(2))

# however, from here we can see that hybrid has too few data to learn reliably
# furthermore, fully remote and on-site show similiar distribution
# decision: low/moderate signal may help the target, further evaluation after model

In [ ]:
# checking how employement type influences the target
plt.figure(figsize=(10, 6))
sns.boxplot(x='employment_type', y='salary_in_usd', data=df)
plt.title('Salary Distribution by Employment Type')
plt.xlabel('Employment Type')
plt.ylabel('Salary in USD')
plt.grid(axis='y', alpha=0.75)
plt.show()

# pt and fl show similiar salary range
# ct shows a generally higher salary

In [ ]:
percentages = df['employment_type'].value_counts(normalize=True) * 100

print(percentages.round(2))

# highly imbalance with too few data from ct, pt and fl to learn reliably
# 98.96% is ft, other employement type are almost invisble and may introduce noise to model
# decision: drop employement_type

In [ ]:
df = df.drop(columns=['employment_type'])

In [ ]:
display(df.head())

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['salary_in_usd'], bins=30, kde=True)
plt.title('Distribution of Salary in USD')
plt.xlabel('Salary in USD')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.show()

In [ ]:
import numpy as np
df['salary_log'] = np.log1p(df['salary_in_usd'])

In [ ]:
display(df.head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(df['salary_log'], bins=30, kde=True)
plt.title('Distribution of Salary')
plt.xlabel('Salary')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.show()

In [ ]:
from ydata_profiling import ProfileReport

profile = ProfileReport(df, title="Pandas Profiling Report")
profile.to_notebook_iframe()

In [ ]:
print("Number of duplicate rows in df:")
display(df.duplicated().sum())

In [ ]:
df = df.drop_duplicates()

In [ ]:
print("Size of df (rows, columns):")
display(df.shape)

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='company_size', y='salary_in_usd', data=df, order=['s', 'm', 'l'])
plt.title('Salary Distribution by Company Size')
plt.xlabel('Company Size')
plt.ylabel('Salary in USD')
plt.grid(axis='y', alpha=0.75)
plt.show()

In [ ]:
print("Distribution of company size:")
display(df['company_size'].value_counts())

# kept as is as diff company size would offer different salary range

In [ ]:
display(df.head())

In [ ]:
print("Distribution of job titles:")
pd.set_option('display.max_rows', None)
display(df['job_title'].value_counts())

In [ ]:
threshold = 0.01

counts = df['job_title'].value_counts(normalize=True)

df['job_title_clean'] = df['job_title'].apply(
    lambda x: x if counts[x] >= threshold else 'other'
)

In [ ]:
df = df.drop(columns=['job_title'])

In [ ]:
print("Distribution of job titles:")
pd.set_option('display.max_rows', None)
display(df['job_title_clean'].value_counts())

In [ ]:
display(df.head())

In [ ]:
df.to_csv('salaries_clean.csv', index=False)

**Data Splitting**

In [ ]:
df_clean = pd.read_csv('salaries_clean.csv')
display(df_clean.head())

In [ ]:
X = df_clean.drop(columns=['salary_in_usd', 'salary_log'])
y = df_clean['salary_log']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
# encode categorical columns
X_train = pd.get_dummies(X_train)
X_test = pd.get_dummies(X_test)

# align column
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

In [ ]:
print(X_train.shape)
print(X_test.shape)